# 01 Importation

In [ ]:
import pandas as pd

In [ ]:
# read the file
df_raw = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/data/Motor_Vehicle_Collisions.csv', low_memory=False)

# Data source : https://catalog.data.gov/dataset/motor-vehicle-collisions-crashes

In [ ]:
# Check the dataframe
df_raw.info()

In [ ]:
df_raw.head()

# 02 DataFrame Exploration

In [ ]:
# Dataframe size
df_raw_size = df_raw.shape[0]
print(f" The raw Dataframe has {df_raw_size:,} rows.")

In [ ]:
# Check Borough column :
missing_borough = df_raw['BOROUGH'].isnull().sum()
print(f"Borough is missing in {missing_borough:,} rows.")
print(f"{round((100*missing_borough/df_raw_size),2)} %")

# 03 How to get back Boroughs from data ?


In [ ]:
# from ZIP code column :
from_ZIP = df_raw.loc[(df_raw['BOROUGH'].isnull()) & (df_raw['ZIP CODE'].notnull())]
print(f" We can find {from_ZIP.shape[0]:,} Borough with the ZIP code column.")

In [ ]:
# from location column :
from_loc = df_raw.loc[(df_raw['BOROUGH'].isnull()) & (df_raw['LOCATION'].notnull())]
print(f" We can find {from_loc.shape[0]:,} Boroughs with the location column.")
print(f"\t{round((100*from_loc.shape[0]/missing_borough), 2)} % of the missing Boroughs.")
print(f"\t{round((100*from_loc.shape[0]/df_raw_size), 2)} % of the raw data.")

In [ ]:
# from Lat & Long columns (if different than location column):
from_latlong = df_raw.loc[(df_raw['BOROUGH'].isnull()) & (df_raw['LOCATION'].isnull()) & (df_raw['LATITUDE'].notnull()) & (df_raw['LONGITUDE'].notnull()) ]
print(f" We can find {from_latlong.shape[0]:,} Boroughs with the latitude and longitude columns.")

In [ ]:
# from streets
from_streets = df_raw.loc[(df_raw['BOROUGH'].isnull()) & (df_raw['LOCATION'].isnull()) & ((df_raw['ON STREET NAME'].notnull())|(df_raw['CROSS STREET NAME'].notnull())|(df_raw['OFF STREET NAME'].notnull()))]
print(f" We can find {from_streets.shape[0]:,} Boroughs with the streets columns.")
print(f"\t{round((100*from_streets.shape[0]/missing_borough), 2)} % of the missing Boroughs.")
print(f"\t{round((100*from_streets.shape[0]/df_raw_size), 2)} % of the raw data.")


In [ ]:
# Location impossible to find :
impossible_location_mask = \
    (df_raw['BOROUGH'].isnull()) & \
    (df_raw['LOCATION'].isnull()) & \
    (df_raw['ON STREET NAME'].isnull()) & \
    (df_raw['CROSS STREET NAME'].isnull()) & \
    (df_raw['OFF STREET NAME'].isnull())

impossible_location = df_raw.loc[impossible_location_mask]
print(f"There are {impossible_location.shape[0]:,} accidents impossible to locate.")
print(f"\t{round((100*impossible_location.shape[0]/missing_borough), 2)} % of the missing Boroughs.")
print(f"\t{round((100*impossible_location.shape[0]/df_raw_size), 2)} % of the raw data.")

From this point 03, we see that the columns 'ZIP CODE', 'LATITUDE', and 'LONGITUDE' are useless. The 'streets' column represents 8% of the raw data. However, the 'streets' columns are very difficult to clean, and some 'drives' are impossible to locate because they are often different streets with the same drive name in New York.
So, I am going to ignore the streets for now.


# 04 Data cleaning and column selection

In [ ]:
# Show the raw DF
df_raw.info()

The focus is made on time, location and human toll.

In [ ]:
# select all but useless column from point 03 and contribution factors and vehicule type.
df = df_raw.drop(df_raw.columns[[3,6]+list(range(7,10))+list(range(18,29))], axis=1)

In [ ]:
# drop the impossible to find
df = df.dropna(subset=['BOROUGH', 'LATITUDE', 'LONGITUDE'], how='all')

In [ ]:
# check the selection
df.info()

In [ ]:
# Merge CRASH DATE and CRASH TIME
df.insert(0, 'Time', df['CRASH DATE'] + ' ' + df['CRASH TIME'])

# Drop CRASH DATE and CRASH TIME
df = df.drop(df.columns[[1,2]], axis=1)

In [ ]:
# Time column in date-time format
import datetime as dt
df['Time'] = pd.to_datetime(df['Time'], format='%m/%d/%Y %H:%M')

In [ ]:
# Renaming
df = df.rename(columns={'BOROUGH':'Borough', 'LATITUDE' : 'Lat','LONGITUDE' : 'Long', 'NUMBER OF PERSONS INJURED' : 'Injured', 'NUMBER OF PERSONS KILLED' : 'Killed', 'NUMBER OF PEDESTRIANS INJURED' : 'Pedest inj', 'NUMBER OF PEDESTRIANS KILLED' : 'Pedest killed', 'NUMBER OF CYCLIST INJURED' : 'Cyclist inj', 'NUMBER OF CYCLIST KILLED': 'Cyclist killed', 'NUMBER OF MOTORIST INJURED' : 'Motorist inj', 'NUMBER OF MOTORIST KILLED' : 'Motorist killed' })

In [ ]:
# Transform the NaN Values in Injured and killed
df['Injured'] = df['Pedest inj'] + df['Cyclist inj'] +df['Motorist inj']
df['Killed'] = df['Pedest killed'] + df['Cyclist killed'] +df['Motorist killed']

In [ ]:
# Change the string in 'Title' style
df['Borough'] = df['Borough'].str.title()

In [ ]:
# check
df.info()

In [ ]:
# Borough NaN -> 'Unknown'
# df_raw['BOROUGH'] = df_raw['BOROUGH'].fillna('Unknown').str.title()

# 04b : Streets cleaning

In [ ]:
# Show the raw DF
df_raw.info()

In [ ]:
df_streets = df_raw.iloc[:,list(range(0,10))]
df_streets = df_streets.drop(columns=["ZIP CODE", "LOCATION"])

In [ ]:
df_streets[df_streets.select_dtypes(['object']).columns] = df_streets.select_dtypes(['object']).apply(lambda x: x.str.title())

In [ ]:
# cleaning function : find streets and remove house numbers

import re
def remove_street_number(street_name):
  if type(street_name) != str :
    return street_name
    # Expression régulière pour trouver le numéro de rue suivi d'espaces
  else :
    return re.sub(r'^\d+-?\d*\s{2,}', '', street_name)


Apply the cleaning function

In [ ]:
df_streets['On street'] = df_streets['ON STREET NAME'].apply(lambda x : remove_street_number(x ))

In [ ]:
df_streets['Cross street'] = df_streets['CROSS STREET NAME'].apply(lambda x : remove_street_number(x ))

In [ ]:
df_streets['Off street'] = df_streets['OFF STREET NAME'].apply(lambda x : remove_street_number(x ))

In [ ]:
# Check the result
df_streets.head(10)

In [ ]:
# drop the original streets
df_streets = df_streets.drop(df_streets.columns[[5,6,7]], axis=1)

In [ ]:
# Check the result
df_streets.head(10)

# 05a Find Boroughs

This step is skipped.
It lasts too long on my computer.

I downloaded the result in a the output.csv file.
So I can continue working and execute the codes faster in this workshop.

In [ ]:
# df.head()

In [ ]:
# import geopandas as gpd
# from shapely.geometry import Point
# nyc_boroughs = gpd.read_file('/content/drive/MyDrive/Colab Notebooks/data/NY Borough Boundaries.geojson')

In [ ]:
# Function to find the borough from coordinates


# def find_borough(long, lat):
#     point = Point(long, lat)
#     for idx, borough in nyc_boroughs.iterrows():
#         if borough['geometry'].contains(point):
#             return borough["boro_name"]
#     return 'Unknown'

In [ ]:
# How many entries to find :

# df.loc[df['Borough'].isnull()==True].info()

In [ ]:
# WARNING : this is long !!

# Apply the function :
# df_nan = df[df['Borough'].isna()]
# df_nan['Borough'] = df_nan.apply(lambda row: find_borough(row['Long'], row['Lat']), axis=1)

# # Update the df
# df.update(df_nan)

In [ ]:
# Check if it worked
# df.loc[df['Borough'].isnull()==True].info()

In [ ]:
# Download the result

# from google.colab import files
# df.to_csv('output.csv', encoding = 'utf-8-sig')
# files.download('output.csv')

# 06 Time analysis

In [ ]:
# read the file
dft = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/data/output.csv', index_col=0)

In [ ]:
# The imported csv, did not keep the Timestamp format of the 'Time' column, fix it :
dft['Time'] = pd.to_datetime(dft['Time'])

In [ ]:
# dft is the readed file from point 05a : to change
dft['Borough'] = dft['Borough'].fillna('Unknown')

In [ ]:
# Check the files size :
print('\nLet\'s check this value with what we expected in the "02 Exploration point" :')
print(f"\n\tThe Dataset had \t\t\t\t{df_raw_size:,} lines.")
print(f"\n\tData dropped : ")
print(f"\t - location from streets had : \t\t\t{from_streets.shape[0]:,} lines.")
print(f"\t - location impossible to find had : \t\t{impossible_location.shape[0]:,} lines.")
print("\t",'-'*65)
print(f'\tThe "DataFrame" dft should have : \t\t{df_raw_size-from_streets.shape[0]-impossible_location.shape[0]:,} lines.')
print(f"\n\tIt actually has : \t\t\t\t{dft.shape[0]:,} lines.")

In [ ]:
# Check 1
dft.head(10)

In [ ]:
# Check 2
dft['Borough'].value_counts()

In [ ]:
# We still have "unknown" : Why ?
dft.loc[dft['Borough']=='Unknown']

My function failed to locate 8,995 rows.

At least 2 of them (index 47 and 212, see above) are located in (0,0).
For the others, I suspect them to be on the boundaries.

I am going to clean the (0,0) locations lines and leave the others.
I will come back to it later on.


In [ ]:
# select and check the lines to clean : latitude = 0.0
lat_zero = dft.loc[(dft['Borough']=='Unknown')&(dft['Lat']==0)]
lat_zero

In [ ]:
# Clean the null latitudes
dft = dft.drop(lat_zero.index)

In [ ]:
# Checks if a value in Long is null
long_zero = dft.loc[(dft['Borough']=='Unknown')&(dft['Long']==0)]
long_zero

In [ ]:
# Check the df.  (Time is in "object type" -> fixed below)
dft.info()

In [ ]:
dft['Time'].min()

In [ ]:
dft['Time'].max()

In [ ]:
# vizualisation importation
import matplotlib.pyplot as plt
import seaborn as sns

GRAPH 1 : Incidents by Hour

In [ ]:
# Make Hour division :
dft['Hour'] = dft['Time'].dt.floor('H').dt.hour

# Group values
by_hour = dft['Hour'].value_counts().sort_index()

# Plot
sns.barplot(x=by_hour.index, y=by_hour.values)

# # Ajouter des titres et libellés
plt.title('Total des incidents par tranche horaire')
plt.xlabel('Heure de la journée')
plt.ylabel('Nombre d\'incidents')

# Configurer
plt.xticks(rotation=45)
plt.grid(axis="y", color='grey')
plt.yticks([20000,40000,60000,80000,100000,120000,140000])


# Afficher le diagramme
plt.tight_layout()

GRAPH 2 : Incidents by day

In [ ]:
#

order = [ 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dft['day_name'] = dft['Time'].dt.day_name().astype(pd.CategoricalDtype(categories=order, ordered=True))

by_day = dft['day_name'].value_counts().sort_index()
sns.barplot(x=by_day.index, y=by_day.values)

# # Ajouter des titres et libellés
plt.title('Total des incidents par jour')
plt.xlabel('')
plt.ylabel('Nombre d\'incidents')

# Configurer l'axe x
plt.xticks(rotation=45)

# # Afficher le diagramme
plt.tight_layout()

GRAPH 3 : Incidents by month

In [ ]:
#

# add
m_order = ['January', 'February', 'March','April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
dft['Month'] = dft['Time'].dt.month_name().astype(pd.CategoricalDtype(categories= m_order, ordered=True))
by_month = dft['Month'].value_counts().sort_index()
sns.barplot(x=by_month.index, y=by_month.values)

# Labels
plt.title('Total des incidents par mois')
plt.xlabel('')
plt.ylabel('Nombre d\'incidents')

# Configure
plt.xticks(rotation=45)

# Print the diagramme
plt.tight_layout()

GRAPH 4 : Evolution of incidents

In [ ]:
#

dft['Year'] = dft['Time'].dt.year

# Créer le graphique à barres
by_year = dft['Year'].value_counts().sort_index()
sns.barplot(x=by_year.index, y=by_year.values, palette="viridis", hue = by_year.index, legend=False)

#  Ajouter des titres et libellés
plt.title('Total des incidents par année')
plt.xlabel('Année')
plt.ylabel('Nombre d\'incidents')

# Configurer l'axe x
plt.xticks(rotation=45)
plt.grid(axis="y", color='grey')
plt.yticks([50000,100000,150000,200000])

# # Afficher le diagramme
plt.tight_layout()

# 07 Borough analysis

In [ ]:
# palette creation
Borough_colour = {'Bronx': 'red', 'Queens': 'blue', 'Brooklyn': 'green', 'Manhattan': 'orange', 'Staten Island': 'purple', 'Unknown': 'grey'}

In [ ]:
# GRAPH 1 : Incidents by Borough
by_Borough = dft['Borough'].value_counts().sort_index()
sns.barplot(x=by_Borough.index, y= by_Borough.values, palette= Borough_colour, hue=by_Borough.index, legend=False)

# labels
plt.title('Total des incidents par comté')
plt.xlabel('Comté')
plt.ylabel('Nombre d\'incidents')

# Configure
plt.xticks(rotation=45)
plt.grid(axis="y", color='grey')

# Print
plt.tight_layout()

In [ ]:
# GRAPH 2 : Death by Borough
by_killed = dft.groupby('Borough')['Killed'].sum()

# by_Borough = dft['Borough'].value_counts().sort_index()
sns.barplot(x = by_killed.index, y = by_killed, palette= Borough_colour, hue=by_Borough.index, legend=False)

# # Ajouter des titres et libellés
plt.title('Total de personnes décédées par comté')
plt.xlabel('Comté')
plt.ylabel('Nombre de personnes décédées')

# Configurer
plt.xticks(rotation=45)
plt.grid(axis="y", color='grey')

# Afficher le diagramme
plt.tight_layout()

In [ ]:
# GRAPH 3 : Injuries by Borough
by_injuries = dft.groupby('Borough')['Injured'].sum()

# by_Boroug = dft['Borough'].value_counts().sort_index()
sns.barplot(x = by_injuries.index, y = by_injuries, palette= Borough_colour, hue=by_Borough.index, legend=False)

# # Ajouter des titres et libellés
plt.title('Total de personnes blessées par comté')
plt.xlabel('Comté')
plt.ylabel('Nombre de personnes blessées')

# Configurer
plt.xticks(rotation=45)
plt.grid(axis="y", color='grey')

# Afficher le diagramme
plt.tight_layout()

# 08 Borough Maps

In [ ]:
import folium

NYmap = folium.Map(location=[40.7128, -74.0060], zoom_start=10)

# Styling
def style_function(feature):
    return {
        'fillColor': Borough_colour[feature['properties']['boro_name']],
        'color': 'black',
        'weight': 2,
        'dashArray': '5, 5',
        'fillOpacity': 0.5
    }

# boroughs limits
nyc_boroughs = '/content/drive/MyDrive/Colab Notebooks/data/NY Borough Boundaries.geojson'
folium.GeoJson(
    nyc_boroughs,
    style_function = style_function,
    tooltip=folium.GeoJsonTooltip(fields=['boro_name'], aliases=['Comté:'])).add_to(NYmap)

In [ ]:
NYmap

# 09 Heat map : persons killed
WARNING : NAN in Lat/Long

In [ ]:
from folium import plugins
from folium.plugins import HeatMap

Heatmap = folium.Map(location=[40.7128, -74.0063], zoom_start=10)

# Killed selection and NaN dropping in Latitude :
df_heat = dft.loc[(df['Killed']>0)&(df['Lat'].notnull())]


In [ ]:
df_heat.info()

In [ ]:
heat_data = [[row['Lat'], row['Long']] for index, row in df_heat.iterrows()]

HeatMap(heat_data, radius=15).add_to(Heatmap)
Heatmap

# XXX Geocode saves

In [ ]:
# CODE TEST : ok

# importing modules
# from geopy.geocoders import Nominatim
# geolocator = Nominatim(user_agent="intersection_finder", timeout=10)
# location1 = geolocator.geocode('45 STREET, New York')
# print(location1.latitude, location1.longitude)

In [ ]:
# Function to find location


# def location_from_streets(row):
#     streetlocator = Nominatim(user_agent="intersection_finder")
#     street = str(row['Off street'])

#     if street[0].isdigit():
#         number, street = street.split(' ', maxsplit=1)
#         address = f"{number} {street}, New York"
#     else:
#         address = f"{street}, New York"

#     coord = streetlocator.geocode(address)

#     if coord:
#         return coord.Lat, coord.longitude
#     else:
#         return None, None


# # df_location[single_Off_street_mask, 'Coord'] = df_location.apply(location_from_streets, axis=1)
# df_location['Coord'][1601] = df_location.apply(location_from_streets, axis=1)


In [ ]:
# Find Borough with long and lat
# TIME OUT

# # importing modules
# from geopy.geocoders import Nominatim

# # calling the nominatim tool
# geoLoc = Nominatim(user_agent="GetLoc")

# def get_Borough(row):
#   if pd.isnull(row['Borough']) and pd.notnull(row['Lat']) and pd.notnull(row['Long']):
#     coord = f"{row['Lat']}, {row['Long']}"
#     locname = geoLoc.reverse(coord)
#     address = locname.raw['address']
#     return address.get('suburb', locname.address)

#   else :
#     return row['Borough']

# df_location['Borough'] = df_location.apply(get_Borough, axis=1)